In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_A8_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_A8_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_A8_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_A8_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 42 trials.
Best value (Accuracy): 0.8494
Best Hyperparameters:
  outer_lr: 0.0004474019395019859
  wd: 0.0001121476480563176
  maml_inner_steps: 15
  maml_alpha_init: 0.010504003538200524
  maml_alpha_init_eval: 0.0034575853439192884
  maml_use_lslr: False
  use_lslr_at_eval: False
  use_maml_msl: hybrid
  maml_msl_num_epochs: 24
  episodes_per_epoch_train: 200
  num_experts: 24
  MOE_top_k: 6
  MOE_gate_temperature: 0.5589950415650072
  MOE_aux_coeff: 0.10309864617499638
  MOE_ctx_out_dim: 32
  MOE_ctx_hidden_dim: 128
  MOE_dropout: 0.12163377838970199
  MOE_aux_loss_plcmt: both


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

#params_to_plot_ARCH = [
#    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
#]

params_to_plot_OPT = [
    "outer_lr", "wd", #"label_smooth", "ft_lr"
]

params_to_plot_MAML = [
    "maml_inner_steps", "maml_alpha_init_eval", "maml_use_lslr", "use_lslr_at_eval", "use_maml_msl", "episodes_per_epoch_train"
]

params_to_plot_MOE_CORE = [
    "num_experts", "MOE_top_k", "MOE_aux_coeff", "MOE_gate_temperature"
]

params_to_plot_MOE_AUX = [
    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout"
]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_MAML)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_CORE)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

In [14]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
35,35,0.849375,None,2026-04-20 08:56:48.802008,0 days 01:38:18.759380
6,6,0.824688,None,2026-04-20 00:53:29.731064,0 days 03:30:41.976686
16,16,0.793229,None,2026-04-20 03:06:59.677252,0 days 05:00:49.731148
27,27,0.776042,None,2026-04-20 06:11:34.767311,0 days 02:35:12.777301
1,1,0.757292,None,2026-04-19 23:37:58.193874,0 days 06:08:00.823386
24,24,0.730938,None,2026-04-20 04:53:10.651760,0 days 03:51:34.059278
33,33,0.721354,None,2026-04-20 08:45:36.092561,0 days 01:45:33.456504
21,21,0.712604,None,2026-04-20 04:26:53.217389,0 days 03:48:48.636867
9,9,0.699062,None,2026-04-20 01:16:50.309775,0 days 03:09:14.351409
19,19,0.680208,None,2026-04-20 04:12:57.706306,0 days 04:43:32.218930


In [ ]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)